In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("noorsaeed/news-categories-dataset",path = 'bbc-text.csv')

print("Path to dataset files:", path)

100%|██████████| 1.83M/1.83M [00:00<00:00, 3.20MB/s]

Extracting zip of bbc-text.csv...
Path to dataset files: /root/.cache/kagglehub/datasets/noorsaeed/news-categories-dataset/versions/1/bbc-text.csv


In [2]:
import pandas as pd
df = pd.read_csv('/root/.cache/kagglehub/datasets/noorsaeed/news-categories-dataset/versions/1/bbc-text.csv')
df.head()


,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


**Splitting Dataset**

In [3]:
from sklearn.model_selection import train_test_split


# Split the DataFrame into train and test sets
df_train, df_test = train_test_split(df, test_size=0.2, shuffle=True, random_state=42)

# Display the first few rows of each DataFrame
print("Train Set:")
print(df_train.shape)

print("\nTest Set:")
print(df_test.shape)

Train Set:
(1780, 2)

Test Set:
(445, 2)


In [4]:

df_train['category'].value_counts()

,count
category,
sport,413
business,409
politics,334
tech,319
entertainment,305


In [5]:
df_test['category'].value_counts()

,count
category,
business,101
sport,98
politics,83
tech,82
entertainment,81


In [6]:
encoded_dict = {"sport":0,"business":1, 'politics':2, "entertainment":3,'tech':4}
df_train['category'] = df_train['category'].map(encoded_dict)

In [7]:
df_test['category'] = df_test['category'].map(encoded_dict)

In [8]:
from keras.utils import to_categorical
# Convert the 'category' column to categorical values
y_train = to_categorical(df_train['category'])
y_test = to_categorical(df_test['category'])
y_train.shape, y_test.shape

((1780, 5), (445, 5))

**Tokenization from the transformers package**

In [9]:
from transformers import AutoTokenizer,TFBertModel
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
bert = TFBertModel.from_pretrained('bert-base-cased')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [10]:
max_len_train = max(len(text) for text in df_train['text'])
max_len_test= max(len(text) for text in df_test['text'])
max_len_train, max_len_test

(19136, 25483)

In [11]:
 # Tokenize the input
max_len = 70
x_train = tokenizer(
    text=df_train.text.tolist(),
    add_special_tokens=True,
    max_length=max_len,
    truncation=True,
    padding=True,
    return_tensors='tf',
    return_token_type_ids=False,
    return_attention_mask=True,
    verbose=True
)
x_test = tokenizer(
    text=df_test.text.tolist(),
    add_special_tokens=True,
    max_length=max_len,
    truncation=True,
    padding=True,
    return_tensors='tf',
    return_token_type_ids=False,
    return_attention_mask=True,
    verbose=True
)

In [12]:
input_ids = x_train['input_ids']
attention_mask = x_train['attention_mask']
print("input ids:",input_ids.shape)
print("attention_mask:", attention_mask.shape)

input ids: (1780, 70)
attention_mask: (1780, 70)


**Implementing the Model**

In [13]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import TruncatedNormal
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Dense

In [14]:
# Define input layers
input_ids = Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
input_mask = Input(shape=(max_len,), dtype=tf.int32, name="attention_mask")

input_ids.shape,input_mask.shape

((None, 70), (None, 70))

In [ ]:
# Define model architecture (assuming 'bert' is already defined)
# Convert Keras Input tensors to TensorFlow tensors before passing to bert
input_ids_tensor = tf.convert_to_tensor(input_ids)
input_mask_tensor = tf.convert_to_tensor(input_mask)

# Pass TensorFlow tensors to bert
embeddings = bert(input_ids_tensor, attention_mask=input_mask_tensor)[0]

out = tf.keras.layers.GlobalMaxPool1D()(embeddings)
out = Dense(128, activation='relu')(out)
out = tf.keras.layers.Dropout(0.1)(out)
out = Dense(32, activation='relu')(out)
y = Dense(5, activation='sigmoid')(out)

# Create the model
model = tf.keras.Model(inputs=[input_ids, input_mask], outputs=y)  # 'input_ids', 'input_mask' are used to define model inputs
model.layers[2].trainable = True

In [ ]:
from keras.optimizers import Adam
from keras.optimizers.schedules import ExponentialDecay
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy

# Define learning rate schedule
initial_learning_rate = 5e-05
lr_schedule = ExponentialDecay(
    initial_learning_rate,
    decay_steps=10000,  # Adjust this value according to your needs
    decay_rate=0.01,    # Adjust this value according to your needs
    staircase=True)
# Define optimizer, loss, and metrics
optimizer = Adam(
    learning_rate=lr_schedule,
    epsilon=1e-08,
    clipnorm=1.0
)
loss = CategoricalCrossentropy(from_logits=True)
metric = CategoricalAccuracy(name='balanced_accuracy')
# Compile the model
model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=metric
)

**Training the Model**

In [ ]:
train_history = model.fit(
    x={'input_ids': x_train['input_ids'], 'attention_mask': x_train['attention_mask']},
    y=y_train,
    validation_data=(
        {'input_ids': x_test['input_ids'], 'attention_mask': x_test['attention_mask']},
        y_test
    ),
    epochs=2,
    batch_size=36
)

In [ ]:
predicted_raw = model.predict({'input_ids':x_test['input_ids'],'attention_mask':x_test['attention_mask']})
predicted_raw[0]

In [ ]:
def predict(text):
    x_val = tokenizer(
        text=texts,
        add_special_tokens=True,
        max_length=70,
        truncation=True,
        padding='max_length',
        return_tensors='tf',
        return_token_type_ids=False,
        return_attention_mask=True,
        verbose=True
    )
    validation = model.predict({'input_ids': x_val['input_ids'], 'attention_mask': x_val['attention_mask']}) * 100

    # Create lists to store the results
    labels = []
    scores = []

    # Iterate over the predicted values and store them
    for key, value in zip(encoded_dict.keys(), validation[0]):
        labels.append(key)
        scores.append(value)

    # Get the predicted label with the highest score
    predicted_label = labels[scores.index(max(scores))]

    # Return the labels, scores, and the predicted label with its score
    return labels, scores, predicted_label, max(scores)

***Implementation***

In [ ]:
texts = 'input the textbrown and blair face new rift claims for the umpteenth time  tony blair and gordon brown are said to have declared all out war on each other.  this time the alleged rift is over who should take the credit for the government s global aid and debt initiatives  particularly in the wake of the tsunami disaster - an issue many hoped and believed was above such things. it dominated the prime minister s monthly news conference  which saw mr blair start in full irritation mode as he was forced to bat away question after question about his relationship with his neighbour. as he told journalists:  i am not interested in what goes in and out of newspapers. there is a complete unity of purpose.  and he again heaped praise on mr brown saying he was doing a great job  and would continue doing it - although he would not commit to any job for mr brown after the election.  so why did he arrange his press conference at the last moment so it coincided with mr brown s long-arranged keynote speech on aid and debt  he was asked  by now mr blair had moved from irritation mode to his barely disguised fury setting. he snapped back that the hacks knew very well what the operational reasons were for the timing of his press conference. well  not really  as it happens.  and he repeated what a great man gordon was and how united they were  before again sneering that he took absolutely no notice of what went in and out of the newspapers  preferring to get on with the job of doing the best for the country and the world. although in the next breath he declared:  i get increasingly alarmed by what i read in the newspapers  before catching himself on and quickly adding:  in so far as i read them of course.  he probably had good reason to be alarmed because the newspapers had been full of stories about the claimed open warfare between the two men.  as far as the timing of the prime minister s press conference is concerned  there are two options. the first is that it was a calculated attempt to upstage the chancellor and seize back the initiative on the big issue of the moment. if that is the case it suggests that even the fear of seriously negative newspaper headlines is not enough to stop the squabbling. the second option is that it was an unavoidable coincidence  which would suggest the government has lost its once-famed ability to strictly co-ordinate announcements - through the infamous downing street grid - to avert just such allegations.  either way  the effect was the same - to overshadow the big announcements of government policy on a hugely pertinent issue. and there had been previous suggestions that the new year had started with a fresh outbreak of the warfare between the two men. firstly  the prime minister insisted on wednesday that he had been intimately involved in the development of the proposals to get g8 countries to freeze debt repayments from the tsunami-hit countries. it was claimed he had been embarrassed by the fact that gordon brown appeared to have taken the initiative over the government s response to the disaster while mr blair was still on holiday in egypt.  then  as if to pour fuel on the flames  both men separately spoke about working on tsunami or wider aid and development policy with their cabinet colleagues foreign secretary jack straw  aid minister hilary benn and deputy prime minister john prescott - without mentioning the other. all this came amid fresh claims that mr brown was still seething that he had been excluded from a prominent role in general election planning and had  as a result  started to set out his own platform. the fact that he used an article in the guardian newspaper to set out what he believed  should  be in the manifesto  has embarked on a mini tour of britain to set out his aid plans and will next week visit africa on the same mission - often seen as the prime minister s  turf  - has only added to the impression of rival camps operating entirely independently of each other. the prime minister denied all that as well  repeating his insistence that it was inconceivable the economy and the chancellor would not be at the centre of the election campaign. but the big fear with many on the labour benches now is that  unless a lid can be put on the speculation over the rivalry  it may even threaten to undermine the election campaign itself.'

labels, scores, predicted_label, pred_prob = predict(texts)

for name, score in zip(labels,scores):
  print("Labels : ", name, " Scores : ", score)

print("====================================================")
print("predicted_label : ", predicted_label, " Scores : ", pred_prob)